# MVP: Автоматическая классификация типов математических ошибок

**Магистерская диссертация Мелконяна О. А.**  
**Программа "Электронный бизнес и цифровые инновации", ВШЭ**

Ноутбук реализует embedding-based ML-классификатор для диагностики типа ошибки в ученическом решении задачи по математике 5–9 классов.

**Запуск.** Открыть в Google Colab, нажать `Runtime → Run all`. Полный прогон занимает 12–15 минут на бесплатном tier.

**Архитектура.** Текст решения → multilingual-e5-large → SVM/RF/LogReg → label + confidence → рекомендация.

## Ячейка 1. Установка зависимостей

In [ ]:
!pip install -q sentence-transformers==2.7.0 scikit-learn==1.4.0 pandas==2.0.3 numpy==1.26.0 matplotlib==3.8.0 seaborn==0.13.0 umap-learn==0.5.5 joblib==1.3.0

import os, random, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import joblib

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

print('Зависимости установлены. Версия pandas:', pd.__version__)

## Ячейка 2. Демо-датасет (мини-выборка для быстрой проверки)

В реальной работе подгружается из Google Drive файл `dataset_247.csv`. Здесь приведены 20 примеров для немедленной демонстрации работы пайплайна без зависимости от внешних файлов.

In [ ]:
demo_data = [
    # Концептуальные ошибки
    ('Найдите сумму дробей 1/3 + 1/4', '1/3 + 1/4 = 2/7', 'concept_error', 'дроби'),
    ('Найдите (-2)^4', '(-2)^4 = -16', 'concept_error', 'степени'),
    ('Вычислите 3/5 + 2/5', '3/5 + 2/5 = 5/10', 'concept_error', 'дроби'),
    ('Найдите квадратный корень из 16', 'корень из 16 = 4 или -4 = ±4', 'concept_error', 'степени'),
    ('Решите x^2 = -4', 'x = корень из -4 = 2', 'concept_error', 'уравнения'),
    
    # Вычислительные ошибки
    ('Решите 3x + 7 = 22', '3x = 22 - 7. 3x = 14. x = 14/3', 'computation_error', 'уравнения'),
    ('Вычислите 17 + 28', '17 + 28 = 35', 'computation_error', 'арифметика'),
    ('Найдите 12 умножить на 8', '12 * 8 = 88', 'computation_error', 'арифметика'),
    ('Решите 5x = 35', '5x = 35. x = 8', 'computation_error', 'уравнения'),
    ('Найдите 0.3 умножить на 0.7', '0.3 * 0.7 = 2.1', 'computation_error', 'дроби'),
    
    # Процедурные ошибки
    ('Решите x^2 = 9', 'x = корень из 9. x = 3', 'procedural_error', 'уравнения'),
    ('Решите x^2 - 5x + 6 = 0', 'D = 25 - 24 = 1. x = 2', 'procedural_error', 'уравнения'),
    ('Решите систему x + y = 5, x - y = 1', 'x + y = 5. 2x = 5 + 1 = 6. x = 3. Подставляю в x + y = 5: y = 2. Проверка x + y = 5: 3 + 2 = 5. Все верно', 'procedural_error', 'уравнения'),
    ('Найдите все корни уравнения x^2 = 25', 'x = корень из 25 = 5', 'procedural_error', 'уравнения'),
    ('Решите неравенство x^2 > 4', 'x > 2', 'procedural_error', 'уравнения'),
    
    # Невнимательность
    ('Найдите периметр прямоугольника со сторонами 3 и 5', 'P = 3 * 5 = 15 см^2', 'carelessness', 'геометрия'),
    ('Решите неравенство x > 4', 'x >= 4', 'carelessness', 'уравнения'),
    ('Найдите площадь квадрата со стороной 6', 'P = 4 * 6 = 24', 'carelessness', 'геометрия'),
    ('Найдите 25% от 80', '25 + 80 = 105', 'carelessness', 'арифметика'),
    ('Решите x - 8 = 12', 'x = 12 - 8 = 4', 'carelessness', 'уравнения'),
]

df = pd.DataFrame(demo_data, columns=['problem_text', 'solution_text', 'error_type', 'fgos_section'])
df['full_text'] = 'query: ' + df['problem_text'] + ' Решение: ' + df['solution_text']

print('Размер датасета:', df.shape)
print('\nРаспределение типов ошибок:')
print(df['error_type'].value_counts())
print('\nПервые 5 примеров:')
df.head()

## Ячейка 3. Загрузка модели multilingual-e5-large и векторизация

In [ ]:
print('Загрузка модели multilingual-e5-large (560 млн параметров, ~90 сек при первом запуске)...')
model = SentenceTransformer('intfloat/multilingual-e5-large')
print('Модель загружена. Размерность эмбеддингов:', model.get_sentence_embedding_dimension())

print('\nВекторизация датасета...')
embeddings = model.encode(df['full_text'].tolist(), batch_size=8, show_progress_bar=True, convert_to_numpy=True)
print('Размерность матрицы эмбеддингов:', embeddings.shape)

# Сохраняем для повторного использования
np.save('/content/embeddings_e5_demo.npy', embeddings)
print('Эмбеддинги сохранены в /content/embeddings_e5_demo.npy')

## Ячейка 4. Стратифицированный train-test split

In [ ]:
y = df['error_type'].values
X = embeddings

# На демо-выборке 20 примеров split 80/20 дает только 4 теста на класс. 
# В полной работе на 247 примерах split дает 50 примеров теста.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f'Train size: {len(X_train)}, Test size: {len(X_test)}')
print(f'\nРаспределение в train:')
print(pd.Series(y_train).value_counts())

## Ячейка 5. Обучение трёх классификаторов

In [ ]:
classifiers = {
    'SVM_RBF': SVC(C=1.0, gamma='scale', kernel='rbf', class_weight='balanced', probability=True, random_state=RANDOM_STATE),
    'RandomForest': RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced', random_state=RANDOM_STATE),
    'LogReg': LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced', multi_class='multinomial', random_state=RANDOM_STATE)
}

trained_models = {}
results = {}

for name, clf in classifiers.items():
    clf.fit(X_train, y_train)
    trained_models[name] = clf
    y_pred = clf.predict(X_test)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    results[name] = {
        'f1_macro': f1_macro,
        'predictions': y_pred,
        'classification_report': classification_report(y_test, y_pred, zero_division=0)
    }
    print(f'\n=== {name} ===')
    print(f'Macro-F1: {f1_macro:.3f}')
    print(results[name]['classification_report'])

# Лучшая модель
best_name = max(results, key=lambda k: results[k]['f1_macro'])
print(f'\nЛучшая модель: {best_name} (F1={results[best_name]["f1_macro"]:.3f})')

## Ячейка 6. Confusion Matrix для лучшей модели

In [ ]:
best_model = trained_models[best_name]
y_pred_best = best_model.predict(X_test)

class_names = sorted(df['error_type'].unique())
cm = confusion_matrix(y_test, y_pred_best, labels=class_names)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, cbar=False)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title(f'Confusion Matrix: {best_name} (Macro-F1={results[best_name]["f1_macro"]:.3f})')
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=120)
plt.show()
print('Сохранено в /content/confusion_matrix.png')

## Ячейка 7. UMAP-визуализация эмбеддингов в 2D

In [ ]:
import umap

reducer = umap.UMAP(n_components=2, random_state=RANDOM_STATE, n_neighbors=5, min_dist=0.3)
embedding_2d = reducer.fit_transform(embeddings)

plt.figure(figsize=(10, 7))
for label in class_names:
    mask = df['error_type'] == label
    plt.scatter(embedding_2d[mask, 0], embedding_2d[mask, 1], label=label, alpha=0.7, s=100)
plt.legend(loc='best')
plt.title('UMAP-проекция эмбеддингов multilingual-e5-large по 4 типам ошибок')
plt.xlabel('UMAP-1')
plt.ylabel('UMAP-2')
plt.tight_layout()
plt.savefig('/content/umap_2d.png', dpi=120)
plt.show()
print('Сохранено в /content/umap_2d.png')

## Ячейка 8. Интерактивная демонстрация (предсказание на произвольном вводе)

In [ ]:
# Словарь рекомендаций по типу ошибки
recommendations = {
    'concept_error': 'Непонимание правила. Повтори определение математического объекта и применяемые к нему свойства.',
    'computation_error': 'Арифметическая ошибка. Метод верный, нужно проверять вычисления на черновике.',
    'procedural_error': 'Сбой в порядке решения. Повтори алгоритм пошагово и разбери два эталонных примера.',
    'carelessness': 'Невнимательность. Перечитывай условие после получения ответа, проверяй единицы измерения и подмены.'
}

user_facing_labels = {
    'concept_error': 'Непонимание правила',
    'computation_error': 'Арифметическая ошибка',
    'procedural_error': 'Сбой в порядке решения',
    'carelessness': 'Невнимательность или описка'
}

def diagnose_error(problem, solution, model=best_model, embedder=model):
    full_text = f'query: {problem} Решение: {solution}'
    emb = embedder.encode([full_text], convert_to_numpy=True)
    
    predicted_label = best_model.predict(emb)[0]
    proba = best_model.predict_proba(emb)[0]
    confidence = float(proba.max())
    
    return {
        'predicted_type_internal': predicted_label,
        'predicted_type_user': user_facing_labels[predicted_label],
        'confidence': round(confidence, 3),
        'recommendation': recommendations[predicted_label],
        'escalate_to_llm': confidence < 0.8
    }

# Тестовые сценарии из § 3.5
test_scenarios = [
    ('Найдите сумму дробей 1/3 + 1/4', '1/3 + 1/4 = 2/7'),
    ('Решите уравнение 3x + 7 = 22', '3x = 22 - 7. 3x = 14. x = 14/3'),
    ('Решите уравнение x^2 = 9', 'x = корень из 9. x = 3'),
    ('Найдите периметр прямоугольника со сторонами 3 и 5', 'P = 3 * 5 = 15 см^2'),
]

print('=' * 70)
print('ДЕМОНСТРАЦИЯ ПАЙПЛАЙНА на 4 сценариях из § 3.5')
print('=' * 70)

for i, (prob, sol) in enumerate(test_scenarios, 1):
    result = diagnose_error(prob, sol)
    print(f'\n--- Сценарий {i} ---')
    print(f'Условие. {prob}')
    print(f'Решение. {sol}')
    print(f'Диагноз. {result["predicted_type_user"]} (confidence={result["confidence"]})')
    print(f'Рекомендация. {result["recommendation"]}')
    if result['escalate_to_llm']:
        print('Низкая confidence. В production-режиме здесь произошла бы эскалация к LLM-fallback.')

## Ячейка 9. Сохранение модели и инструкции по запуску демо в Streamlit

In [ ]:
joblib.dump(best_model, '/content/best_classifier.joblib')
with open('/content/recommendations.json', 'w', encoding='utf-8') as f:
    json.dump({'recommendations': recommendations, 'user_facing_labels': user_facing_labels}, f, ensure_ascii=False, indent=2)

print('Модель и словари сохранены.')
print('Для запуска Streamlit-демо локально:')
print('  1. Скачать best_classifier.joblib и recommendations.json')
print('  2. Создать app.py с функцией diagnose_error выше')
print('  3. Запустить streamlit run app.py')
print('\nПолный код Streamlit-приложения в репозитории работы:')
print('\n=== Прогон ноутбука завершён ===')